In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/score.py
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/cache/index/embeddings.npy
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/cache/index/doc_ids.json
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/src/test_metrics.py
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/src/metrics.py
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/src/build_index.py
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/src/test_llm.py
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/src/run_closed_book.py
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/src/run_oracle.py
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/src/test_retrieval.py
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/src/run_dense_rag.py
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/src/compare_with_score.py
/kaggle/input/datasets/foroughkoohi/graphrag-qa-data/src/explore_data.py
/kaggle/input/datasets/foroughkoohi/graphrag-qa-da

In [ ]:
# ==== Cell 1: Environment setup ====
!pip uninstall -y vllm torch torchaudio torchvision transformers
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu129
!pip install vllm==0.11.0
!pip install "transformers>=4.40.0,<5.0.0"
!pip install sentence-transformers accelerate

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:


In [3]:
# ==== Cell 2: Verify installs ====
import torch, torchaudio, vllm, transformers
print(f"torch:        {torch.__version__}")
print(f"torchaudio:   {torchaudio.__version__}")
print(f"vllm:         {vllm.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


torch:        2.8.0+cu129
torchaudio:   2.8.0+cu129
vllm:         0.11.0
transformers: 4.57.6
CUDA available: True


In [4]:
# ==== Cell 3: Copy course pack to /kaggle/working ====
import shutil
from pathlib import Path

SRC = Path("/kaggle/input/datasets/foroughkoohi/graphrag-qa-data")
DST = Path("/kaggle/working")
for item in SRC.iterdir():
    dst = DST / item.name
    if item.is_dir():
        shutil.copytree(item, dst, dirs_exist_ok=True)
    else:
        shutil.copy(item, dst)

!ls -la /kaggle/working

total 32
drwxr-xr-x 6 root root 4096 Oct 17 14:14 .
drwxr-xr-x 5 root root 4096 Oct 17 13:54 ..
drwxr-xr-x 3 root root 4096 Oct 17 13:44 cache
drwxr-xr-x 3 root root 4096 Oct 17 13:44 data
-rw-r--r-- 1 root root 8120 Oct 17 14:14 score.py
drwxr-xr-x 2 root root 4096 Oct 17 13:44 src
drwxr-xr-x 2 root root 4096 Oct 17 13:54 .virtual_documents


In [5]:
# ==== Cell 4: Common imports and config ====
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import json
import time
import yaml
from pathlib import Path

PACK = Path("/kaggle/working/data/course_pack_v1")
INDEX_DIR = Path("/kaggle/working/cache/index")
MODEL_PATH = "/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1"
K = 10  # <-- FIX: was 5

# Shared prompt: strict, identical for dev and hidden
PROMPT_TEMPLATE = """Answer the question with the SHORTEST possible answer.
CRITICAL RULES:
- If the answer is an entity, give ONLY the entity name
- If the answer is a date, give ONLY the date
- If the answer is a place, give ONLY the place name
- If it's a yes/no question, answer exactly 'yes' or 'no'
- DO NOT write sentences
- DO NOT explain
- DO NOT start with "based on" or "the context"
- Use lowercase
- No punctuation

{context_block}Question: {question}
Answer:"""

def write_config(out_dir, system_name, split, description, embedder, retrieval_unit, retrieval_k, command):
    cfg = {
        "system_name": system_name,
        "phase": 1,
        "split": split,
        "description": description,
        "models": {
            "llm": {
                "name": "alphadua/qwen3-8b-awq-community",
                "base_model": "Qwen/Qwen3-8B",
                "quantization": "awq",
                "where": "Kaggle T4, vLLM 0.11.0",
                "temperature": 0,
            },
            "embedder": embedder,
            "reranker": None,
        },
        "settings": {
            "retrieval": {"unit": retrieval_unit, "k": retrieval_k},
            "max_new_tokens": 30,
            "prompt_template": "strict short-answer",
        },
        "command": command,
        "date": time.strftime("%Y-%m-%d"),
    }
    with open(out_dir / "run_config.yaml", "w", encoding="utf-8") as f:
        yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print(f"Config ready. k = {K}")


Config ready. k = 10


In [6]:
# ==== Cell 5: Load LLM once and reuse ====
from vllm import LLM, SamplingParams

print(f"Loading LLM: {MODEL_PATH}")
llm = LLM(
    model=MODEL_PATH,
    quantization="awq",
    dtype="half",
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    enforce_eager=True,
)
sampling = SamplingParams(temperature=0, max_tokens=30)
print("LLM ready")

INFO 10-17 14:14:26 [__init__.py:216] Automatically detected platform cuda.
Loading LLM: /kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1
INFO 10-17 14:14:47 [utils.py:233] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': '/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1'}
INFO 10-17 14:15:07 [model.py:547] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 10-17 14:15:07 [model.py:1510] Using max model len 4096
INFO 10-17 14:15:08 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 10-17 14:15:08 [__init__.py:381] Cudagraph is disabled under eager mode
WARNING 10-17 14:15:09 [__init__.py:3036] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 10-17 14:15:19 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=321) INFO 10-17 14:15:22 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=321) INFO 10-17 14:15:22 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1', speculative_config=None, tokenizer='/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1', skip_

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:32<00:32, 32.41s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:41<00:00, 18.41s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:41<00:00, 20.51s/it]
(EngineCore_DP0 pid=321) 


(EngineCore_DP0 pid=321) INFO 10-17 14:16:09 [default_loader.py:267] Loading weights took 41.03 seconds
(EngineCore_DP0 pid=321) INFO 10-17 14:16:10 [gpu_model_runner.py:2653] Model loading took 5.7071 GiB and 41.283731 seconds
(EngineCore_DP0 pid=321) INFO 10-17 14:16:17 [gpu_worker.py:298] Available KV cache memory: 5.92 GiB
(EngineCore_DP0 pid=321) INFO 10-17 14:16:17 [kv_cache_utils.py:1087] GPU KV cache size: 43,072 tokens
(EngineCore_DP0 pid=321) INFO 10-17 14:16:17 [kv_cache_utils.py:1091] Maximum concurrency for 4,096 tokens per request: 10.52x
(EngineCore_DP0 pid=321) WARNING 10-17 14:16:17 [cudagraph_dispatcher.py:106] cudagraph dispatching keys are not initialized. No cudagraph will be used.
(EngineCore_DP0 pid=321) INFO 10-17 14:16:18 [core.py:210] init engine (profile, create kv cache, warmup model) took 7.91 seconds
(EngineCore_DP0 pid=321) INFO 10-17 14:16:18 [__init__.py:381] Cudagraph is disabled under eager mode
INFO 10-17 14:16:19 [llm.py:306] Supported_tasks: ['gene

In [7]:
# ==== Cell 6: Load embedder and index (used for dense) ====
import numpy as np
from sentence_transformers import SentenceTransformer

embeddings = np.load(INDEX_DIR / "embeddings.npy")
with open(INDEX_DIR / "doc_ids.json", encoding="utf-8") as f:
    doc_ids = json.load(f)
print(f"Index loaded: {embeddings.shape}")

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("Embedder ready")

corpus = {
    p["doc_id"]: p["text"]
    for p in (json.loads(l) for l in open(PACK / "corpus.jsonl", encoding="utf-8") if l.strip())
}


Index loaded: (3882, 384)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder ready


In [8]:
# ==== Helper functions ====
def clean_answer(text):
    return text.strip().split("\n")[0].strip().rstrip(".").lower()

def retrieve_top_k(questions, k=K):
    q_texts = [q["question"] for q in questions]
    q_embs = embedder.encode(q_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    scores = q_embs @ embeddings.T
    top_idx = np.argsort(-scores, axis=1)[:, :k]
    return [[doc_ids[i] for i in row] for row in top_idx]

def build_prompts(questions, retrieved_ids=None, gold_map=None):
    """Build prompts for closed_book (retrieved_ids=None, gold_map=None),
       dense (retrieved_ids provided), or oracle (gold_map provided)."""
    prompts = []
    for q in questions:
        if gold_map is not None:
            rids = gold_map[q["qid"]]["gold_doc_ids"]
        elif retrieved_ids is not None:
            rids = retrieved_ids[len(prompts)]
        else:
            rids = []
        if rids:
            ctx = "\n\n".join(f"- {corpus[did]}" for did in rids) + "\n\nContext:\n"
            ctx = "Context:\n" + "\n\n".join(f"- {corpus[did]}" for did in rids) + "\n\n"
        else:
            ctx = ""
        prompts.append(PROMPT_TEMPLATE.format(context_block=ctx, question=q["question"]))
    return prompts

def save_predictions(out_dir, questions, answers, retrieved_lists):
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "predictions.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for q, a, rids in zip(questions, answers, retrieved_lists):
            f.write(json.dumps({
                "qid": q["qid"],
                "answer": a,
                "retrieved": rids,
            }, ensure_ascii=False) + "\n")
    return path

print("Helpers ready")

Helpers ready


In [9]:
# ==== Cell 7: CLOSED-BOOK on dev ====
split = "dev"
questions = [json.loads(l) for l in open(PACK / f"questions_{split}.jsonl", encoding="utf-8") if l.strip()]
print(f"{split} questions: {len(questions)}")

prompts = build_prompts(questions)  # no context
t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"closed-book {split}: {time.time()-t0:.1f}s")

answers = [clean_answer(o.outputs[0].text) for o in outputs]
out = Path(f"/kaggle/working/runs/{split}/closed_book")
path = save_predictions(out, questions, answers, [[] for _ in questions])
write_config(out, "closed_book", split,
             "Qwen3-8B AWQ via vLLM, no retrieval, strict prompt",
             embedder=None, retrieval_unit=None, retrieval_k=0,
             command=f"Kaggle notebook: closed-book on {split}")
print(f"Saved: {path}")

dev questions: 300


Adding requests:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

closed-book dev: 138.2s
Saved: /kaggle/working/runs/dev/closed_book/predictions.jsonl


In [10]:
# ==== Cell 8: CLOSED-BOOK on hidden ====
split = "hidden"
questions = [json.loads(l) for l in open(PACK / f"questions_{split}.jsonl", encoding="utf-8") if l.strip()]
print(f"{split} questions: {len(questions)}")

prompts = build_prompts(questions)
t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"closed-book {split}: {time.time()-t0:.1f}s")

answers = [clean_answer(o.outputs[0].text) for o in outputs]
out = Path(f"/kaggle/working/runs/{split}/closed_book")
path = save_predictions(out, questions, answers, [[] for _ in questions])
write_config(out, "closed_book", split,
             "Qwen3-8B AWQ via vLLM, no retrieval, strict prompt",
             embedder=None, retrieval_unit=None, retrieval_k=0,
             command=f"Kaggle notebook: closed-book on {split}")
print(f"Saved: {path}")

hidden questions: 300


Adding requests:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

closed-book hidden: 79.7s
Saved: /kaggle/working/runs/hidden/closed_book/predictions.jsonl


In [ ]:
# ==== Cell 9: DENSE on dev (k=10) ====
split = "dev"
questions = [json.loads(l) for l in open(PACK / f"questions_{split}.jsonl", encoding="utf-8") if l.strip()]

retrieved_ids = retrieve_top_k(questions, k=K)
lengths = sorted(set(len(r) for r in retrieved_ids))
print(f"retrieved lengths: {lengths}")  # should be [10]
assert lengths == [K], f"BUG: lengths={lengths}, expected [{K}]"

prompts = build_prompts(questions, retrieved_ids=retrieved_ids)
t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"dense {split}: {time.time()-t0:.1f}s")

answers = [clean_answer(o.outputs[0].text) for o in outputs]
out = Path(f"/kaggle/working/runs/{split}/dense")
path = save_predictions(out, questions, answers, retrieved_ids)
write_config(out, "dense", split,
             f"bge-small dense retrieval top-{K}, Qwen3-8B AWQ via vLLM, strict prompt",
             embedder="BAAI/bge-small-en-v1.5", retrieval_unit="passage", retrieval_k=K,
             command=f"Kaggle notebook: dense on {split}")
print(f"Saved: {path}")

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

retrieved lengths: [10]


Adding requests:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
# ==== Cell 10: DENSE on hidden (k=10) ====
split = "hidden"
questions = [json.loads(l) for l in open(PACK / f"questions_{split}.jsonl", encoding="utf-8") if l.strip()]

retrieved_ids = retrieve_top_k(questions, k=K)
lengths = sorted(set(len(r) for r in retrieved_ids))
print(f"retrieved lengths: {lengths}")  # should be [10]
assert lengths == [K], f"BUG: lengths={lengths}, expected [{K}]"

prompts = build_prompts(questions, retrieved_ids=retrieved_ids)
t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"dense {split}: {time.time()-t0:.1f}s")

answers = [clean_answer(o.outputs[0].text) for o in outputs]
out = Path(f"/kaggle/working/runs/{split}/dense")
path = save_predictions(out, questions, answers, retrieved_ids)
write_config(out, "dense", split,
             f"bge-small dense retrieval top-{K}, Qwen3-8B AWQ via vLLM, strict prompt",
             embedder="BAAI/bge-small-en-v1.5", retrieval_unit="passage", retrieval_k=K,
             command=f"Kaggle notebook: dense on {split}")
print(f"Saved: {path}")


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

retrieved lengths: [10]


Adding requests:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [13]:
# ==== Cell 11: ORACLE on dev ====
split = "dev"
questions = [json.loads(l) for l in open(PACK / f"questions_{split}.jsonl", encoding="utf-8") if l.strip()]
labels = {l["qid"]: l for l in (json.loads(l) for l in open(PACK / f"labels_{split}.jsonl", encoding="utf-8") if l.strip())}

gold_lists = [labels[q["qid"]]["gold_doc_ids"] for q in questions]
lengths = sorted(set(len(g) for g in gold_lists))
print(f"oracle gold lengths: {lengths}")  # should be [1, 2, 4]

prompts = build_prompts(questions, gold_map=labels)
t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"oracle {split}: {time.time()-t0:.1f}s")

answers = [clean_answer(o.outputs[0].text) for o in outputs]
out = Path(f"/kaggle/working/runs/{split}/oracle")
path = save_predictions(out, questions, answers, gold_lists)
write_config(out, "oracle", split,
             "gold_doc_ids as context, Qwen3-8B AWQ via vLLM, strict prompt",
             embedder=None, retrieval_unit="oracle", retrieval_k="all gold",
             command=f"Kaggle notebook: oracle on {split}")
print(f"Saved: {path}")


oracle gold lengths: [2, 4]


Adding requests:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

oracle dev: 822.3s
Saved: /kaggle/working/runs/dev/oracle/predictions.jsonl


In [14]:
# ==== Cell 12: Free GPU memory before scoring ====
import gc, torch
del llm
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

[rank0]:[W1017 16:08:05.042148707 ProcessGroupNCCL.cpp:1538] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


GPU free: 15.33 GB / 15.64 GB


In [15]:
# ==== Cell 13: Validate all prediction files ====
print("=" * 60)
print("VALIDATION")
print("=" * 60)
for split in ["dev", "hidden"]:
    for sys in ["closed_book", "dense", "oracle"]:
        p = Path(f"/kaggle/working/runs/{split}/{sys}/predictions.jsonl")
        if not p.exists():
            print(f"{split}/{sys}: NOT FOUND (ok if oracle on hidden)")
            continue
        rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
        lengths = sorted(set(len(r["retrieved"]) for r in rows))
        cfg = Path(f"/kaggle/working/runs/{split}/{sys}/run_config.yaml")
        print(f"{split}/{sys}: {len(rows)} rows, retrieved lengths={lengths}, config={'OK' if cfg.exists() else 'MISSING'}")

VALIDATION
dev/closed_book: 300 rows, retrieved lengths=[0], config=OK
dev/dense: 300 rows, retrieved lengths=[10], config=OK
dev/oracle: 300 rows, retrieved lengths=[2, 4], config=OK
hidden/closed_book: 300 rows, retrieved lengths=[0], config=OK
hidden/dense: 300 rows, retrieved lengths=[10], config=OK
hidden/oracle: NOT FOUND (ok if oracle on hidden)


In [16]:
# ==== Cell 14: Score dev systems ====
!python /kaggle/working/score.py /kaggle/working/runs/dev/closed_book/predictions.jsonl --pack /kaggle/working/data/course_pack_v1 --split dev

# %%
!python /kaggle/working/score.py /kaggle/working/runs/dev/dense/predictions.jsonl --pack /kaggle/working/data/course_pack_v1 --split dev

# %%
!python /kaggle/working/score.py /kaggle/working/runs/dev/oracle/predictions.jsonl --pack /kaggle/working/data/course_pack_v1 --split dev

valid: /kaggle/working/runs/dev/closed_book/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2           0.0   [  0.0,   0.0]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5           0.0   [  0.0,   0.0]
  hitall@5           0.0   [  0.0,   0.0]
  recall@10          0.0   [  0.0,   0.0]
  hitall@10          0.0   [  0.0,   0.0]
  mrr                0.0   [  0.0,   0.0]
  ndcg@10            0.0   [  0.0,   0.0]
  em                25.7   [ 21.0,  31.0]
  f1                29.7   [ 24.8,  35.0]

type: bridge_comparison (n=66)
  recall@2           0.0   [  0.0,   0.0]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5           0.0   [  0.0,   0.0]
  hitall@5           0.0   [  0.0,   0.0]
  recall@10          0.0   [  0.0,   0.0]
  hitall@10          0.0   [  0.0,   0.0]
  mrr                0.0   [  0.0,   0.0]
  ndcg@10            0.0   [  0.0,   0.0]
  em                45.5   [ 33.3,  56.1]
  f1                46.8   [ 34.9,  57.5]

type: comparison (n=73)
  reca

In [17]:
# ==== Cell 15: Paired comparisons ====
!python /kaggle/working/score.py /kaggle/working/runs/dev/dense/predictions.jsonl --pack /kaggle/working/data/course_pack_v1 --split dev --compare /kaggle/working/runs/dev/closed_book/predictions.jsonl

# %%
!python /kaggle/working/score.py /kaggle/working/runs/dev/oracle/predictions.jsonl --pack /kaggle/working/data/course_pack_v1 --split dev --compare /kaggle/working/runs/dev/dense/predictions.jsonl


valid: /kaggle/working/runs/dev/dense/predictions.jsonl (300 predictions)
valid: /kaggle/working/runs/dev/closed_book/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2          62.7   [ 59.7,  65.4]
  hitall@2          28.0   [ 22.7,  33.3]
  recall@5          71.5   [ 68.6,  74.4]
  hitall@5          43.0   [ 37.7,  49.0]
  recall@10         74.7   [ 71.8,  77.4]
  hitall@10         47.7   [ 42.0,  53.3]
  mrr               97.7   [ 96.1,  98.9]
  ndcg@10           77.1   [ 75.0,  79.2]
  em                35.3   [ 30.0,  40.7]
  f1                41.3   [ 36.3,  46.7]

type: bridge_comparison (n=66)
  recall@2          46.2   [ 43.9,  48.1]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5          52.3   [ 50.0,  54.9]
  hitall@5           1.5   [  0.0,   4.5]
  recall@10         54.5   [ 51.9,  57.2]
  hitall@10          1.5   [  0.0,   4.5]
  mrr               97.7   [ 94.7, 100.0]
  ndcg@10           65.0   [ 62.7,  67.3]
  em                50.0   [ 37.9,  62.1]

In [18]:
# ==== Cell 16: Zip everything and provide download link ====
import shutil
from IPython.display import FileLink

# Only zip what's needed (predictions + configs)
zpath = shutil.make_archive("/kaggle/working/runs_clean", "zip", "/kaggle/working/runs")
print(f"Created: {zpath}")
FileLink("runs_clean.zip")

Created: /kaggle/working/runs_clean.zip


/kaggle/working/runs_clean.zip